In [14]:
import os
import pickle
from PIL import Image
from tqdm import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from facenet_pytorch import InceptionResnetV1

In [ ]:
def load_and_transform_image(image_path, augment=False, save_augmented=False, save_dir="augmented_images", label=None):
    """
    Memuat gambar dari path dan menerapkan transformasi yang diperlukan, termasuk augmentasi jika diaktifkan.
    
    Args:
        image_path (str): Path ke file gambar.
        augment (bool): Jika True, terapkan augmentasi pada gambar.
        
    Returns:
        Tensor: Gambar yang telah diubah menjadi tensor.
    """
    augment_transform = [
        transforms.Resize((256, 256)),  # Ukuran input untuk InceptionResnetV1
    ]
    
    if augment:
        # Menambahkan augmentasi: Random Horizontal Flip dan ColorJitter
        augment_transform.extend([
            transforms.ColorJitter(brightness=0.7, contrast=0, saturation=0, hue=0),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(degrees=(-6.5, 6.5))
        ])
    
    # Transformasi lengkap untuk pengolahan data (dengan normalisasi)
    transform_list = augment_transform + [
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ]
    
    transform_for_save = transforms.Compose(augment_transform + [transforms.ToTensor()])
    transform_for_processing = transforms.Compose(transform_list)
    
    try:
        image = Image.open(image_path).convert('RGB')
        
        if save_augmented and augment:
            label_dir = os.path.join(save_dir, label)
            if not os.path.exists(label_dir):
                os.makedirs(label_dir)
            
            # Simpan gambar yang sudah diaugmentasi
            augmented_image_tensor = transform_for_save(image)
            augmented_image_path = os.path.join(label_dir, os.path.basename(image_path))
            augmented_image = transforms.ToPILImage()(augmented_image_tensor)  # Convert back to PIL Image
            augmented_image.save(augmented_image_path)  # Simpan gambar augmentasi

        # Kembalikan gambar yang telah diaugmentasi dan dinormalisasi
        transformed_image = transform_for_processing(image)
        
        return transformed_image
    except Exception as e:
        print(f"Error memuat gambar {image_path}: {e}")
        return torch.zeros(3, 256, 256)

In [16]:
class FacesDataset(Dataset):
    def __init__(self, root_dir, transform=None, augment=False, save_augmented=False):
        """
        Args:
            root_dir (str): Direktori root yang berisi subfolder untuk setiap label.
            transform (callable, optional): Transformasi yang akan diterapkan pada gambar.
            augment (bool): Jika True, terapkan augmentasi pada gambar.
        """
        self.root_dir = root_dir
        self.transform = transform
        self.augment = augment
        self.save_augmented = save_augmented
        self.image_paths = []
        self.labels = []
        
        # Enumerasi semua subfolder dan gambar di dalamnya
        for label in os.listdir(root_dir):
            label_dir = os.path.join(root_dir, label)
            if not os.path.isdir(label_dir):
                continue
            for img_name in os.listdir(label_dir):
                if img_name.lower().endswith('.ppm'):
                    full_path = os.path.join(label_dir, img_name)
                    if os.path.isfile(full_path):
                        self.image_paths.append(full_path)
                        self.labels.append(label)
                    else:
                        print(f"File tidak ditemukan: {full_path}")
        
        if len(self.image_paths) == 0:
            raise RuntimeError(f"Tidak ada file .ppm ditemukan di {root_dir}")
                    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        if self.transform:
            image = self.transform(img_path)
        else:
            image = load_and_transform_image(img_path, augment=self.augment, save_augmented=self.save_augmented, label=label)
        return image, label

In [17]:
def generate_embeddings(train_dir, batch_size=32, device='cuda' if torch.cuda.is_available() else 'cpu'):
    """
    Menghasilkan embedding untuk semua gambar dalam direktori train, menggunakan data asli dan augmentasi.
    
    Args:
        train_dir (str): Path ke direktori train.
        batch_size (int, optional): Ukuran batch untuk DataLoader.
        device (str, optional): Device untuk menjalankan model ('cuda' atau 'cpu').
        
    Returns:
        dict: Dictionary dengan label sebagai key dan list embedding asli serta augmentasi sebagai value.
    """
    embeddings_dict = {}

    try:
        # Dataset dan DataLoader untuk data asli dan augmentasi
        original_dataset = FacesDataset(root_dir=train_dir, augment=False, save_augmented=False)
        augmented_dataset = FacesDataset(root_dir=train_dir, augment=True, save_augmented=True)
        
        original_dataloader = DataLoader(original_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
        augmented_dataloader = DataLoader(augmented_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    except RuntimeError as e:
        print(e)
        return embeddings_dict

    # Inisialisasi model FaceNet pra-terlatih
    model = InceptionResnetV1(pretrained='vggface2').eval().to(device)

    # Membuat embedding untuk data asli dan augmentasi
    with torch.no_grad():
        # Loop untuk data asli
        for images, labels in tqdm(original_dataloader, desc="Membuat Embedding Data Asli"):
            images = images.to(device)
            embeddings = model(images).cpu().numpy()
            
            for label, embedding in zip(labels, embeddings):
                if label not in embeddings_dict:
                    embeddings_dict[label] = []
                embeddings_dict[label].append(embedding)

        # Loop untuk data augmentasi
        for images, labels in tqdm(augmented_dataloader, desc="Membuat Embedding Data Augmentasi"):
            images = images.to(device)
            embeddings = model(images).cpu().numpy()
            
            for label, embedding in zip(labels, embeddings):
                embeddings_dict[label].append(embedding)

    return embeddings_dict

In [ ]:
def main():
    train_directory = 'train'  # Ganti dengan path ke direktori train Anda
    
    if not os.path.exists(train_directory):
        print(f"Direktori train tidak ditemukan: {train_directory}")
        return
    
    embeddings = generate_embeddings(train_directory)
    
    if embeddings:
        # Simpan embeddings ke file pickle
        with open('embeddings_augmentation.pkl', 'wb') as f:
            pickle.dump(embeddings, f)
        
        print("Embedding berhasil disimpan dalam embeddings_augmentation.pkl")
    else:
        print("Tidak ada embedding yang disimpan karena terjadi error.")

In [19]:
main()

Membuat Embedding Data Augmentasi: 100%|██████████| 13/13 [00:35<00:00,  2.71s/it]

Embedding berhasil disimpan dalam embeddings_augment7.pkl
